<a href="https://colab.research.google.com/github/Ravneet-kaur1102/DataScienceTraining/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit plotly pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.4 MB/s eta 0:00:00


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ----------------- PAGE CONFIGURATION -----------------
st.set_page_config(page_title="Retail Performance Dashboard", layout="wide")
st.title("🛍️ Retail Sales & Profit Analytics Dashboard")
st.markdown("Monitor revenue trends, regional performance, and product margins in real-time.")

# ----------------- MOCK DATA GENERATION -----------------
@st.cache_data
def load_data():
    np.random.seed(42)
    dates = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
    categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Beauty & Health', 'Books']
    regions = ['North America', 'Europe', 'APAC', 'LATAM']

    data = {
        'Date': np.random.choice(dates, 1000),
        'Category': np.random.choice(categories, 1000),
        'Region': np.random.choice(regions, 1000),
        'Sales': np.random.uniform(15, 1200, 1000).round(2),
        'Quantity': np.random.randint(1, 10, 1000)
    }
    df = pd.DataFrame(data)
    # Profit calculation with category variance
    df['Profit'] = (df['Sales'] * np.random.uniform(0.1, 0.45, 1000)).round(2)
    df['Date'] = pd.to_datetime(df['Date'])
    return df.sort_values('Date')

df = load_data()

# ----------------- SIDEBAR FILTERS -----------------
st.sidebar.header("Filter Options")

# Date Filter
min_date, max_date = df['Date'].min().to_pydatetime(), df['Date'].max().to_pydatetime()
start_date, end_date = st.sidebar.date_input("Select Date Range", [min_date, max_date])

# Category Filter
all_categories = ['All'] + list(df['Category'].unique())
selected_category = st.sidebar.selectbox("Product Category", all_categories)

# Region Filter
all_regions = ['All'] + list(df['Region'].unique())
selected_region = st.sidebar.selectbox("Geographic Region", all_regions)

# Apply Filters
filtered_df = df[(df['Date'] >= pd.to_datetime(start_date)) & (df['Date'] <= pd.to_datetime(end_date))]

if selected_category != 'All':
    filtered_df = filtered_df[filtered_df['Category'] == selected_category]

if selected_region != 'All':
    filtered_df = filtered_df[filtered_df['Region'] == selected_region]

# ----------------- KPI METRICS ROW -----------------
total_sales = filtered_df['Sales'].sum()
total_profit = filtered_df['Profit'].sum()
total_units = filtered_df['Quantity'].sum()
avg_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0

kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric(label="Total Revenue", value=f"${total_sales:,.2f}")
kpi2.metric(label="Total Profit", value=f"${total_profit:,.2f}")
kpi3.metric(label="Units Sold", value=f"{total_units:,}")
kpi4.metric(label="Avg Profit Margin", value=f"{avg_margin:.2f}%")

st.markdown("---")

# ----------------- VISUALIZATIONS -----------------
col1, col2 = st.columns(2)

with col1:
    st.subheader("📈 Revenue & Profit Trends Over Time")
    # Resample to Monthly for a cleaner trend line
    trend_df = filtered_df.set_index('Date').resample('M').sum().reset_index()

    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(x=trend_df['Date'], y=trend_df['Sales'], name='Revenue', line=dict(color='royalblue', width=3)))
    fig_trend.add_trace(go.Scatter(x=trend_df['Date'], y=trend_df['Profit'], name='Profit', line=dict(color='firebrick', width=3, dash='dash')))
    fig_trend.update_layout(margin=dict(l=20, r=20, t=20, b=20), height=350, legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    st.plotly_chart(fig_trend, use_container_width=True)

with col2:
    st.subheader("🌍 Regional Revenue Distribution")
    region_df = filtered_df.groupby('Region')['Sales'].sum().reset_index()
    fig_region = px.pie(region_df, values='Sales', names='Region', hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
    fig_region.update_layout(margin=dict(l=20, r=20, t=20, b=20), height=350)
    st.plotly_chart(fig_region, use_container_width=True)

col3, col4 = st.columns([6, 4])

with col3:
    st.subheader("📊 Category Sales Breakdown")
    cat_df = filtered_df.groupby('Category')[['Sales', 'Profit']].sum().reset_index().sort_values(by='Sales', ascending=True)
    fig_cat = px.bar(cat_df, x='Sales', y='Category', orientation='h', text_auto='.2s', color='Profit', color_continuous_scale='Viridis')
    fig_cat.update_layout(margin=dict(l=20, r=20, t=20, b=20), height=350)
    st.plotly_chart(fig_cat, use_container_width=True)

with col4:
    st.subheader("📋 Top Transactions Raw Data")
    st.dataframe(
        filtered_df[['Date', 'Category', 'Region', 'Sales', 'Profit']]
        .sort_values(by='Sales', ascending=False)
        .head(100),
        height=320,
        use_container_width=True
    )

Overwriting app.py


In [ ]:
# 1. Get your public IP address (Streamlit requires this security step for tunnel connections)
!wget -qO- ipv4.icanhazip.com

# 2. Expose the server using localtunnel
!npx localtunnel --port 8501 & streamlit run app.py

34.125.63.216
/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦⠧npm warn exec The following package was not found and will be installed: localtunnel@2.0.2
⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://eleven-dryers-kick.loca.lt
